# DEPRECATED -- Shift-Invariant Learned Multiplier (Failure Mode)

**This notebook is kept as a documented failure mode, not as a usable model.** It is cited from `paper/main.tex` (Section 6, "Shift-invariant learned multiplier") and from `my_dfum_criteo_topk.ipynb`'s header as the concrete origin of that lesson. The working, corrected model is in `my_dfum_criteo_topk.ipynb`.

## What this notebook tried to do

An early draft of the top-K decision loss tried to learn the budget threshold `lambda_k` directly, as a trainable `tf.Variable`, and used it inside a softmax:

```python
lambda_k = tf.Variable(0, dtype=tf.float32)   # cell below: hyperparameters
...
scores = tau_hat - self.lambda_k              # cell below: CustomEndpointLayer.call
probabilities = tf.exp(scores) / tf.reduce_sum(tf.exp(scores))   # softmax(scores)
```

The intent: gradient descent would learn the right `lambda_k` for each budget fraction `k`, the same way it learns any other weight.

## Why it fails

`softmax` is **shift-invariant**: `softmax(x) = softmax(x + c)` for any constant `c` added to every element of `x` (the shift cancels between numerator and denominator of `exp(x_i) / sum(exp(x_j))`). Because `lambda_k` is subtracted from *every* element of `tau_hat` before the softmax, `softmax(tau_hat - lambda_k)` is **numerically identical for every value of `lambda_k`**. The loss only touches `lambda_k` through this softmax output, so the loss's gradient with respect to `lambda_k` is exactly zero, everywhere. Gradient descent cannot move `lambda_k` off its initial value (`0`, set in the hyperparameter cell below) -- it sits frozen for the entire training run.

**This does not crash or raise a warning.** Training runs normally, the loss decreases (the rest of the network trains fine), and the printed AUUC below (~0.79) even looks like a plausible number. The only way to catch this is to explicitly check whether `lambda_k`'s value ever changed from its initialization -- nothing in the training curves or final metrics reveals it on their own. That is the actual lesson: **a differentiable-looking construction can have an exactly-zero gradient by construction**, and the failure is silent.

## The fix

`paper/main.tex` (Section 3.2) derives `lambda*` in closed form instead, from the Lagrangian dual of the top-K selection LP: `lambda* = ` the `K`-th largest value of `tau_hat`, an order statistic. It is computed directly each batch via `tf.math.top_k` and treated as a constant (`tf.stop_gradient`) -- never a trainable parameter, which sidesteps this failure mode by construction rather than requiring it to be diagnosed. See `my_dfum_criteo_topk.ipynb`.


In [1]:
import sys
sys.path.append("..")

import pandas as pd

SAVE_DIR = "../data"

file_criteo = SAVE_DIR + "/criteo-uplift-v2.1.csv"

df_criteo = pd.read_csv(file_criteo, sep=',')
df_criteo

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13979587,26.297764,10.059654,9.006250,4.679882,10.280525,4.115453,-3.282109,4.833815,3.839578,13.190056,5.300375,-0.168679,1,0,0,0
13979588,12.642207,10.679513,8.214383,-1.700105,10.280525,3.013064,-13.955150,6.269026,3.971858,13.190056,5.300375,-0.168679,1,0,0,1
13979589,12.976557,10.059654,8.381868,0.842442,11.029584,4.115453,-8.281971,4.833815,3.779212,23.570168,6.169187,-0.168679,1,0,1,0
13979590,24.805064,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0


In [2]:
random_state=20220720
df_criteo=df_criteo.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

X = df_criteo[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']].values

In [3]:
import numpy as np
# scale the feature values between 0 and 1
def scaling(x, min, max):
    return np.where(x < min, 0.0, np.where(x > max, 1.0, (x - min) / (max - min)))

X[:, 0] = scaling(X[:, 0], min=np.min(X[:, 0]), max=np.max(X[:, 0]))
X[:, 1] = scaling(X[:, 1], min=np.min(X[:, 1]), max=np.max(X[:, 1]))
X[:, 2] = scaling(X[:, 2], min=np.min(X[:, 2]), max=np.max(X[:, 2]))
X[:, 3] = scaling(X[:, 3], min=np.min(X[:, 3]), max=np.max(X[:, 3]))
X[:, 4] = scaling(X[:, 4], min=np.min(X[:, 4]), max=np.max(X[:, 4]))
X[:, 5] = scaling(X[:, 5], min=np.min(X[:, 5]), max=np.max(X[:, 5]))
X[:, 6] = scaling(X[:, 6], min=np.min(X[:, 6]), max=np.max(X[:, 6]))
X[:, 7] = scaling(X[:, 7], min=np.min(X[:, 7]), max=np.max(X[:, 7]))
X[:, 8] = scaling(X[:, 8], min=np.min(X[:, 8]), max=np.max(X[:, 8]))
X[:, 9] = scaling(X[:, 9], min=np.min(X[:, 9]), max=np.max(X[:, 9]))
X[:, 10] = scaling(X[:, 10], min=np.min(X[:, 10]), max=np.max(X[:, 10]))
X[:, 11] = scaling(X[:, 11], min=np.min(X[:, 11]), max=np.max(X[:, 11]))

In [4]:
T = df_criteo['treatment'].values.reshape(-1, 1)
Y_visit = df_criteo['visit'].values.reshape(-1, 1)
Y_conv = df_criteo['conversion'].values.reshape(-1, 1)

T.shape, Y_visit.shape, Y_conv.shape

((13979592, 1), (13979592, 1), (13979592, 1))

In [5]:
batch_size = int(len(X) * 0.7)

X_train = X[:batch_size, :]
T_train = T[:batch_size, :]
Y_visit_train = Y_visit[:batch_size, :]
Y_conv_train = Y_conv[:batch_size, :]

X_test = X[batch_size:, :]
T_test = T[batch_size:, :]
Y_visit_test = Y_visit[batch_size:, :]
Y_conv_test = Y_conv[batch_size:, :]

batch_size, X_train.shape, X_test.shape, T_train.shape

(9785714, (9785714, 12), (4193878, 12), (9785714, 1))

In [6]:
sys.path.append("..")
#from model.uplift_model import *
# here in uplift_model, there're self-defined methods include S-Learner, X-Learner, uplift_rank

In [7]:
import matplotlib.pyplot as plt

def plot_loss(history, *losses):
    for loss in losses:
        plt.plot(history.history[loss], label=loss)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()


import causalml
from causalml.metrics import *
import matplotlib.pyplot as plt


def get_causalml_auuc(Y, T, ite_pred, normalize=True):

    metric_df = pd.DataFrame([ite_pred.flatten(),
                               Y.flatten(),
                               T.flatten()]).T


    metric_df.columns=['model','y','w'] 
    uplift_rank_lift = get_cumlift(metric_df)

    normalize = True

    uplift_rank_gain = uplift_rank_lift.mul(uplift_rank_lift.index.values, axis=0)
    if normalize:
        uplift_rank_gain = uplift_rank_gain.div(np.abs(uplift_rank_gain.iloc[-1, :]), axis=1)
    uplift_rank_auuc_score = uplift_rank_gain.sum()/uplift_rank_gain.shape[0]
    
    print(uplift_rank_auuc_score)
    
    step = len(T) // 200 
    
    uplift_rank_gain.iloc[::step, :].plot()
    plt.show()
    
    return uplift_rank_auuc_score, uplift_rank_gain.iloc[::step, :]

  from .autonotebook import tqdm as notebook_tqdm


In [8]:
count = 20

#### Model architecture and decision loss

In [19]:
import tensorflow as tf
from tensorflow import keras

class shape(keras.Layer):
    def call(self, x):
        return tf.shape(x)
    
class CustomEndpointLayer(keras.layers.Layer):
    def __init__(self, alpha, lambda_k, k_values, name=None):
        super().__init__(name=name)
        self.alpha = alpha
        self.lambda_k = lambda_k
        self.k_values = k_values

    def call(self, inputs):
        y_true, selected_output, t1_y_pred, t0_y_pred = inputs

        # Prediction Loss
        #pl_loss = tf.reduce_mean(tf.square(y_true - selected_output))
        pl_loss = tf.keras.losses.BinaryCrossentropy()(y_true, selected_output)

        # Decision Loss
        tau_hat = t1_y_pred - t0_y_pred
        dl_loss = 0.0
        for k in self.k_values:
            k = tf.cast(k, dtype=tf.float32)
            # BUG: softmax(x) is shift-invariant (softmax(x) == softmax(x + c)
            # for any constant c), and self.lambda_k is subtracted from every
            # element here -- so this softmax's output, and therefore the loss,
            # is numerically identical for every value of self.lambda_k. Its
            # gradient is exactly zero; gradient descent can never move it off
            # its initial value. See this notebook's header cell.
            scores = tau_hat - self.lambda_k
            probabilities = tf.exp(scores) / tf.reduce_sum(tf.exp(scores))  # softmax(scores)
            expected_uplift = tf.reduce_sum(probabilities * tau_hat)
            dl_loss += (-expected_uplift - self.lambda_k * k) / k

        total_loss = self.alpha * pl_loss + dl_loss
        self.add_loss(total_loss)

        #tf.print("y_true:", y_true)
        #tf.print("selected_output:", selected_output)
        #tf.print("pl_loss", pl_loss)

        return selected_output, t1_y_pred, t0_y_pred

In [20]:
import tensorflow as tf
from keras import layers, regularizers, Model
from keras.layers import Input, Dense, Lambda


def DFUMModel(alpha, lambda_k, k_values):
    input_x = Input(shape=(12,), name="features")
    input_t = Input(shape=(1,), name = "treatments")
    y_true = Input(shape=(1,), name = "y_true")

    zero_input = Lambda(lambda x: tf.zeros((x, 1), dtype=tf.float32), name='zero_input')(shape()(input_x)[0])
    one_input = Lambda(lambda x: tf.ones((x, 1), dtype=tf.float32), name='one_input')(shape()(input_x)[0])

    x_1 = layers.concatenate([input_x, one_input])
    x_0 = layers.concatenate([input_x, zero_input])

    head0 = Dense(8, activation='relu', name='head0', kernel_regularizer=regularizers.l2(1e-5))(x_0)
    t0_y_pred = Dense(1, name='t0_y_pred', activation='sigmoid', kernel_regularizer=regularizers.l2(1e-5))(head0)

    head1 = Dense(8, activation='relu', name='head1', kernel_regularizer=regularizers.l2(1e-5))(x_1)
    t1_y_pred = Dense(1, name='t1_y_pred', activation='sigmoid', kernel_regularizer=regularizers.l2(1e-5))(head1)

    selected_output = layers.Lambda(lambda x: tf.where(tf.equal(x[0], 1), x[1], x[2]), name='selected_output')([input_t, t1_y_pred, t0_y_pred])

    endpoint_layer = CustomEndpointLayer(alpha, lambda_k, k_values, name='custom_endpoint')([y_true, selected_output, t1_y_pred, t0_y_pred])

    DFUM_model = Model(inputs=[input_x, input_t, y_true], outputs=endpoint_layer)

    return DFUM_model


In [21]:
# Model and training hyperparameters
import os

batch_size = 326888
k_values = [int(0.01 * batch_size), int(0.05 * batch_size), int(0.1 * batch_size), int(0.2 * batch_size), int(0.25 * batch_size), int(0.5 * batch_size)]
lambda_k = tf.Variable(0, dtype=tf.float32)  # never actually trains -- see header cell
alpha = tf.constant(0.8, dtype=tf.float32)
learning_rate = 0.01
epochs = 200

model_save_dir = "../model_file/uplift/criteo/final_model/my_dfum/v3_total_batch_300k_epoch_200/"
os.makedirs(model_save_dir, exist_ok=True)

In [22]:
final_model = DFUMModel(alpha, lambda_k, k_values)
final_model.compile(optimizer='adam')

final_model.summary()

Model: "functional_41"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features            │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shape_83 (shape)    │ (2)               │          0 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shape_82 (shape)    │ (2)               │          0 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_83         │ ()                │          0 │ shape_83[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_82         │ ()                │          0 │ shape_82[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ one_input (Lambda)  │ (None, 1)         │          0 │ get_item_83[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_input (Lambda) │ (None, 1)         │          0 │ get_item_82[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_82      │ (None, 13)        │          0 │ features[0][0],   │
│ (Concatenate)       │                   │            │ one_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_83      │ (None, 13)        │          0 │ features[0][0],   │
│ (Concatenate)       │                   │            │ zero_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head1 (Dense)       │ (None, 8)         │        112 │ concatenate_82[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head0 (Dense)       │ (None, 8)         │        112 │ concatenate_83[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ treatments          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ t1_y_pred (Dense)   │ (None, 1)         │          9 │ head1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ t0_y_pred (Dense)   │ (None, 1)         │          9 │ head0[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ y_true (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ selected_output     │ (None, 1)         │          0 │ treatments[0][0], │
│ (Lambda)            │                   │            │ t1_y_pred[0][0],  │
│                     │                   │            │ t0_y_pred[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom_endpoint     │ [(None, 1),       │          0 │ y_true[0][0],     │
│ (CustomEndpointLay… │ (None, 1), (None, │            │ selected_output[… │
│                     │ 1)]               │            │ t1_y_pred[0][0],  │
│                     │                   │            │ t0_y_pred[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 242 (968.00 B)

 Trainable params: 242 (968.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training loop
from keras.callbacks import ModelCheckpoint

for iteration in range(count):
    print(f"\n--- Iteration {iteration + 1} ---")
    final_model = DFUMModel(alpha, lambda_k, k_values)
    final_model.compile(optimizer='adam')

    mcp_save = ModelCheckpoint(
        os.path.join(model_save_dir, f'dfum_{iteration+1}.weights.h5'),
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        save_weights_only=True
    )

    history = final_model.fit(
        [X_train, T_train, Y_visit_train],
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        validation_split=0.2,
        verbose=1,
        callbacks=[mcp_save]
    )

    plot_loss(history, "loss", "val_loss")
  


In [ ]:
# Evaluate every saved weight file and collect AUUC results
from sklearn import metrics

all_auuc_scores = []

for i in range(count):
    print(f"iteration = {i + 1}")

    model_file = os.path.join(model_save_dir, f'dfum_{i+1}.weights.h5')

    final_model = DFUMModel(alpha, lambda_k, k_values)
    final_model.load_weights(model_file)

    y_pred_test_all = final_model.predict([X_test, T_test, Y_visit_test])
    y_pred_test_all = np.array(y_pred_test_all)
    selected_output = y_pred_test_all[0]
    print("AUC: ", metrics.roc_auc_score(Y_visit_test, selected_output))
    print("MSE: ", metrics.mean_squared_error(Y_visit_test, selected_output))

    uplift_pred_test = y_pred_test_all[1] - y_pred_test_all[2]

    auuc_score = get_causalml_auuc(Y=Y_visit_test, T=T_test, ite_pred=uplift_pred_test)
    all_auuc_scores.append(auuc_score)

all_auuc_scores

In [18]:
import numpy as np

def get_auuc_scores(causalml_auuc_list): 
    auuc_scores = []
    for x in causalml_auuc_list:
        auuc_scores.append(x[0].iloc[0])
    return np.array(auuc_scores)

def print_auuc_res(model_name, auuc_scores):
    print("model: ", model_name)
    
    print("auuc list: ", auuc_scores)
    print("auuc mean: ", np.mean(auuc_scores))
    print("auuc variance: ", np.var(auuc_scores))
    print("auuc standard deviation: ", np.std(auuc_scores))
    
    print()

list = get_auuc_scores(all_auuc_scores)
print("auuc mean: ", np.mean(list))
print("auuc variance: ", np.var(list))
print("auuc standard deviation: ", np.std(list))


auuc mean:  0.790834444008691
auuc variance:  0.0028444285287883226
auuc standard deviation:  0.05333318412384847
